# TCGA-BRCA Minimal Cohort V1 Review

This notebook is review-only. It loads the latest saved minimal cohort v1 outputs from disk, checks the run-level validation state, reviews included and excluded fields, and writes review tables for human audit.


## Load the latest saved minimal cohort v1 build


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'cohort'
    / 'tcga_brca_minimal_cohort_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest minimal cohort v1 pointer not found: {latest_pointer_path}. Run the build script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
cohort_path = repo_root / latest_pointer['minimal_cohort_v1_tsv']
spec_path = repo_root / latest_pointer['minimal_cohort_v1_spec_tsv']
join_audit_path = repo_root / latest_pointer['minimal_cohort_v1_join_audit_tsv']
exclusions_path = repo_root / latest_pointer['minimal_cohort_v1_exclusions_tsv']
summary_path = repo_root / latest_pointer['minimal_cohort_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


## Load saved cohort v1 artifacts


In [ ]:
cohort_df = read_tsv(cohort_path)
spec_df = read_tsv(spec_path)
join_audit_df = read_tsv(join_audit_path)
exclusions_df = read_tsv(exclusions_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError(f"Cohort v1 build is not completed: status={run_log.get('status')}")
if not run_log.get('validation', {}).get('passed', False):
    raise ValueError('Cohort v1 build validation did not pass. Review the saved run_log.json before continuing.')

print(f"Cohort v1 build ID: {latest_pointer['cohort_v1_build_id']}")
print(f"Dry-run build ID: {latest_pointer['dry_run_build_id']}")
print(f"Blueprint run ID: {latest_pointer['blueprint_run_id']}")
print(f"Processed cohort TSV: {cohort_path}")
print(f"Spec TSV: {spec_path}")
print(f"Join audit TSV: {join_audit_path}")
print(f"Exclusions TSV: {exclusions_path}")
print(f"Summary TSV: {summary_path}")
print(f"Run log: {run_log_path}")

validation_df = pd.DataFrame([run_log.get('validation', {})])
display(validation_df)


## Review included fields, exclusions, row counts, and carried ambiguity flags


In [ ]:
def parse_json_list(value: str) -> list[str]:
    if value == '':
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError(f'Expected a JSON list, received: {value}')
    return [str(item) for item in parsed]


cohort_sorted_df = (
    cohort_df.assign(
        provisional_patient_row_id_numeric=pd.to_numeric(
            cohort_df['provisional_patient_row_id'],
            errors='raise',
        )
    )
    .sort_values('provisional_patient_row_id_numeric')
    .drop(columns='provisional_patient_row_id_numeric')
    .reset_index(drop=True)
)
spec_review_df = spec_df.sort_values(['include_in_v1', 'field_category', 'source_table', 'field_name'], ascending=[False, True, True, True]).reset_index(drop=True)
join_review_df = (
    join_audit_df.assign(join_step_order_numeric=pd.to_numeric(join_audit_df['join_step_order'], errors='raise'))
    .sort_values('join_step_order_numeric')
    .drop(columns='join_step_order_numeric')
    .reset_index(drop=True)
)
exclusions_review_df = exclusions_df.sort_values(['exclusion_category', 'source_table', 'entity_value']).reset_index(drop=True)
summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)

included_baseline_fields_df = spec_review_df[(spec_review_df['field_category'] == 'included_baseline_field') & (spec_review_df['include_in_v1'] == 'yes')].reset_index(drop=True)
included_endpoint_fields_df = spec_review_df[(spec_review_df['field_category'] == 'included_endpoint_candidate_field') & (spec_review_df['include_in_v1'] == 'yes')].reset_index(drop=True)
included_biospecimen_anchor_fields_df = spec_review_df[(spec_review_df['field_category'] == 'included_biospecimen_sample_anchor_field') & (spec_review_df['include_in_v1'] == 'yes')].reset_index(drop=True)
excluded_fields_df = spec_review_df[spec_review_df['include_in_v1'] == 'no'].reset_index(drop=True)

ambiguity_flags_review_df = (
    cohort_sorted_df[
        [
            'cohort_v1_build_id',
            'provisional_patient_row_id',
            'bcr_patient_barcode',
            'bcr_patient_uuid',
            'ambiguity_note_flags_json',
        ]
    ]
    .assign(ambiguity_note_flag=lambda df: df['ambiguity_note_flags_json'].map(parse_json_list))
    .explode('ambiguity_note_flag')
    .drop(columns='ambiguity_note_flags_json')
    .reset_index(drop=True)
)

display(summary_review_df)
display(included_baseline_fields_df)
display(included_endpoint_fields_df)
display(included_biospecimen_anchor_fields_df)
display(excluded_fields_df.head(25))


## Save review tables


In [ ]:
cohort_preview_df = cohort_sorted_df.head(100).reset_index(drop=True)

cohort_preview_path = results_root / '67_minimal_cohort_v1_preview.tsv'
spec_review_path = results_root / '68_minimal_cohort_v1_spec.tsv'
join_audit_review_path = results_root / '69_minimal_cohort_v1_join_audit.tsv'
exclusions_review_path = results_root / '70_minimal_cohort_v1_exclusions.tsv'
ambiguity_flags_review_path = results_root / '71_minimal_cohort_v1_ambiguity_flags.tsv'
summary_review_path = results_root / '72_minimal_cohort_v1_summary.tsv'

cohort_preview_df.to_csv(cohort_preview_path, sep='\t', index=False)
spec_review_df.to_csv(spec_review_path, sep='\t', index=False)
join_review_df.to_csv(join_audit_review_path, sep='\t', index=False)
exclusions_review_df.to_csv(exclusions_review_path, sep='\t', index=False)
ambiguity_flags_review_df.to_csv(ambiguity_flags_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f'Saved: {cohort_preview_path}')
print(f'Saved: {spec_review_path}')
print(f'Saved: {join_audit_review_path}')
print(f'Saved: {exclusions_review_path}')
print(f'Saved: {ambiguity_flags_review_path}')
print(f'Saved: {summary_review_path}')

display(cohort_preview_df)
display(summary_review_df)


This notebook remains review-only. It does not parse raw files, build a new cohort from raw sources, freeze the final endpoint, expand child biospecimen layers, or perform modeling.
